# CR-FIQA 기반 조건부 threshold calibration

이 노트북은 완료된 SurvFace Step-4 run(기본값: ArcFace)을 **읽기 전용 입력**으로 사용해 다음 세 가지를 비교합니다.

1. `global_empirical`: 기존 calibration 전체에서 구한 전역 threshold
2. `global_safe`: calibration 내부 fit/safety 분할을 거친 보수적 전역 threshold
3. `fiqa_2bin_conservative_shrunk_safe`: CR-FIQA Low/High 그룹별 부분 풀링 + held-out safety threshold

Saliency의 연구상 위치는 바꾸지 않습니다.

- **1차 목적:** 원본 FR 모델의 공간적 인식 근거가 압축에 따른 embedding distortion, score/rank 변화, threshold crossing과 어떻게 연관되는지 분석
- **2차 목적:** FIQA만으로 설명되지 않는 threshold 불안정성을 saliency가 추가로 설명하는지 검증

현재 SurvFace saliency는 test probe에만 존재하므로, 1차 분석은 가능하지만 FIQA+Saliency threshold 학습은 calibration saliency가 확보될 때까지 누수 방지 게이트가 차단합니다. 기존 공통 orchestration/report 노트북은 수정하지 않습니다.

In [ ]:
# 0. 프로젝트 경로와 공통 import
from __future__ import annotations

import gc
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'research').is_dir() and (candidate / '.git').exists():
            return candidate
    raise RuntimeError('C:\\ronbun 프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.experiments.fiqa_threshold_calibration import (
    assess_saliency_incremental_readiness,
    join_fiqa_score_artifacts,
    load_calibration_comparison_artifact,
    load_condition_score_artifact,
    load_saliency_primary_diagnostics,
    replay_survface_adc_condition_scores,
    run_global_vs_fiqa_calibration,
    write_calibration_comparison_artifact,
    write_condition_score_artifact,
)
from research.fiqa import (
    CRFIQA_VARIANTS,
    infer_cr_fiqa_scores,
    load_cr_fiqa,
    load_fiqa_score_artifact,
    materialize_aligned_bundle_score_artifact,
)
from research.runtime.hashing import sha256_file

PROJECT_ROOT

## 1. 실험 계약

- **주 실험:** CR-FIQA(S), Low/High 2분위, PQ `m=128`, ADC exhaustive search
- **민감도 분석:** 주 실험이 끝난 뒤 CR-FIQA(L)만 동일 설정으로 반복
- FIQA cutpoint와 모든 threshold는 calibration에서만 결정하며 test 결과를 보지 않습니다.
- ADC의 점수공간은 `negative_squared_l2_adc`이며 원본 cosine threshold를 재사용하지 않습니다.
- Low/High group 표본 부족 시 전역 threshold로 fallback하고, 충분한 경우에도 전역값과 부분 풀링합니다.
- `high_saliency`/`low_saliency` mask는 intervention/faithfulness 분석용입니다. `random`은 음성 대조군이며 threshold feature로 사용하지 않습니다.
- 완료 run과 checkpoint는 덮어쓰지 않습니다. 모든 결과 쓰기는 별도 플래그로 명시합니다.

In [ ]:
# 2. 사용자가 조정하는 유일한 설정 셀
SURVFACE_RUN_CANDIDATES = {
    'arcface': PROJECT_ROOT / 'runs' / 'survface_20260829' / (
        '20260829-R001-81149bc4_step4_survface_arcface-7972a704552df378345f'
    ),
    'adaface': PROJECT_ROOT / 'runs' / 'survface_20260830' / (
        '20260830-R001-ec6e5d4a_step4_survface_adaface-4df25b75e065b0b9ed43'
    ),
}
SOURCE_MODEL = 'arcface'
assert SOURCE_MODEL in SURVFACE_RUN_CANDIDATES, 'SOURCE_MODEL은 명시된 완료 run 후보여야 합니다.'
SOURCE_RUN_DIR = SURVFACE_RUN_CANDIDATES[SOURCE_MODEL]
ALIGNED_BUNDLE_DIR = PROJECT_ROOT / 'data' / 'interim' / 'step4' / 'survface' / 'aligned_112'
FIQA_CHECKPOINTS = {
    'S': PROJECT_ROOT / 'models' / 'fiqa' / 'CR-FIQA(S).pth',
    'L': PROJECT_ROOT / 'models' / 'fiqa' / 'CR-FIQA(L).pth',
}
FIQA_VARIANT = 'S'
FIQA_BATCH_SIZE = 64
FIQA_SHARD_SIZE = 8192
MODEL_SMOKE_VARIANTS = ('S', 'L')
MODEL_SMOKE_SAMPLE_COUNT = FIQA_BATCH_SIZE

COMPRESSION_PROFILE = 'pq_512_m128_b8'
SEARCH_MODE = 'pq_adc_exhaustive'
TARGET_FPIRS = (0.01, 0.05, 0.10, 0.20, 0.30)
QUALITY_BIN_COUNT = 2
SHRINKAGE_STRENGTH = 200.0
MINIMUM_GROUP_NON_MATED = 100
SAFETY_FRACTION = 0.30
PARTITION_SEED = 42

RESULT_ROOT = PROJECT_ROOT / 'results' / 'calibration'
VERIFY_CHECKPOINT_HASHES = True
OVERWRITE_OUTPUTS = False

# 긴 계산 및 쓰기는 기본적으로 꺼져 있습니다. 아래 실행 순서를 따르십시오.
RUN_MODEL_SMOKE = False
RUN_FIQA_INFERENCE = False
WRITE_FIQA_ARTIFACT = False
RUN_SCORE_REPLAY = False
WRITE_SCORE_ARTIFACT = False
RUN_THRESHOLD_CALIBRATION = False
WRITE_CALIBRATION_ARTIFACT = False
RUN_PRIMARY_SALIENCY_SUMMARY = True
RUN_SALIENCY_READINESS_CHECK = False

assert FIQA_VARIANT in FIQA_CHECKPOINTS
assert QUALITY_BIN_COUNT == 2, '주 분석은 사전 지정된 Low/High 2분위를 사용합니다.'

### 권장 실행 순서

각 단계가 성공한 뒤 설정 셀의 해당 플래그만 켜고 위에서부터 다시 실행합니다.

1. `RUN_MODEL_SMOKE=True` — S/L checkpoint strict-load 및 production batch(기본 64장) GPU 추론
2. `RUN_FIQA_INFERENCE=True`, `WRITE_FIQA_ARTIFACT=True` — 전체 aligned bundle의 CR-FIQA(S) 점수 생성
3. `RUN_SCORE_REPLAY=True`, `WRITE_SCORE_ARTIFACT=True` — 저장되지 않았던 calibration PQ-ADC 검색만 재생하고 기존 threshold와 일치 여부 감사
4. `RUN_THRESHOLD_CALIBRATION=True`, `WRITE_CALIBRATION_ARTIFACT=True` — Global/FIQA 비교 산출
5. 필요할 때만 `FIQA_VARIANT='L'`로 바꾸어 민감도 분석 반복

실행이 중단되면 이미 완료된 artifact를 SHA-256 검증 후 불러오므로 앞 단계를 다시 계산할 필요가 없습니다.

In [ ]:
# 3. 경량 preflight: 입력 존재, 모델 크기/hash, run 및 aligned contract
def read_json_object(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(payload, dict):
        raise ValueError(f'JSON object가 아닙니다: {path}')
    return payload


if not SOURCE_RUN_DIR.is_dir() or not (SOURCE_RUN_DIR / 'COMPLETED').is_file():
    raise FileNotFoundError(f'완료된 source run이 없습니다: {SOURCE_RUN_DIR}')
source_manifest = read_json_object(SOURCE_RUN_DIR / 'run_manifest.json')
if source_manifest.get('status') != 'completed':
    raise ValueError('source run status가 completed가 아닙니다.')
if source_manifest.get('config', {}).get('dataset_id') != 'survface':
    raise ValueError('이 노트북의 첫 구현은 SurvFace 전용입니다.')
source_model_uid = str(source_manifest.get('config', {}).get('model_uid', ''))
if not source_model_uid:
    raise ValueError('source run에 model_uid가 없습니다.')

aligned_manifest = read_json_object(ALIGNED_BUNDLE_DIR / 'bundle_manifest.json')
contract = aligned_manifest.get('array_contract', {})
if not (
    contract.get('dtype') == 'uint8'
    and contract.get('layout') == 'nhwc'
    and contract.get('color_order') == 'rgb'
    and contract.get('image_size') == [112, 112]
):
    raise ValueError('CR-FIQA 입력은 uint8 NHWC RGB 112x112 aligned bundle이어야 합니다.')

checkpoint_rows = []
for variant, checkpoint in FIQA_CHECKPOINTS.items():
    spec = CRFIQA_VARIANTS[variant]
    if not checkpoint.is_file():
        raise FileNotFoundError(f'CR-FIQA({variant}) checkpoint가 없습니다: {checkpoint}')
    actual_sha256 = sha256_file(checkpoint) if VERIFY_CHECKPOINT_HASHES else None
    checkpoint_rows.append(
        {
            'variant': variant,
            'architecture': spec.architecture,
            'bytes_match': checkpoint.stat().st_size == spec.expected_bytes,
            'sha256_match': actual_sha256 == spec.expected_sha256 if actual_sha256 else 'not_checked',
            'model_uid': spec.model_uid,
            'license': spec.license_id,
        }
    )
checkpoint_preflight = pd.DataFrame(checkpoint_rows)
if not checkpoint_preflight['bytes_match'].all():
    raise ValueError('checkpoint byte size가 등록된 공식 파일과 다릅니다.')
if VERIFY_CHECKPOINT_HASHES and not checkpoint_preflight['sha256_match'].all():
    raise ValueError('checkpoint SHA-256이 등록된 공식 파일과 다릅니다.')

display(checkpoint_preflight)
display(
    pd.DataFrame(
        [{
            'source_run_id': source_manifest['run_id'],
            'dataset_id': source_manifest['config']['dataset_id'],
            'model_uid': source_manifest['config']['model_uid'],
            'aligned_rows': aligned_manifest.get('counts', {}).get('aligned'),
            'selected_fiqa_variant': FIQA_VARIANT,
        }]
    )
)

## 4. 선택 단계 A — CR-FIQA S/L GPU smoke test

공식 구조와 state dict를 `strict=True`로 불러오며 CUDA가 없으면 CPU로 조용히 전환하지 않고 중단합니다. 점수는 sigmoid/min-max 변환 없이 checkpoint의 raw scalar 그대로 사용합니다.

In [ ]:
smoke_summary = pd.DataFrame()
if RUN_MODEL_SMOKE:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError('CUDA smoke test가 요청됐지만 torch.cuda.is_available()이 False입니다.')
    aligned_index = pd.read_csv(
        ALIGNED_BUNDLE_DIR / 'aligned_index.csv',
        nrows=MODEL_SMOKE_SAMPLE_COUNT,
        usecols=['sample_id', 'aligned_face_index'],
    )
    aligned_faces = np.load(
        ALIGNED_BUNDLE_DIR / 'aligned_faces.npy', mmap_mode='r', allow_pickle=False
    )
    face_indices = aligned_index['aligned_face_index'].to_numpy(dtype=np.int64)
    smoke_faces = np.asarray(aligned_faces[face_indices])
    smoke_rows = []
    for variant in MODEL_SMOKE_VARIANTS:
        torch.cuda.reset_peak_memory_stats()
        model, spec = load_cr_fiqa(
            FIQA_CHECKPOINTS[variant], variant=variant, device='cuda',
            verify_official_hash=True,
        )
        scores = infer_cr_fiqa_scores(
            model, smoke_faces, batch_size=MODEL_SMOKE_SAMPLE_COUNT, device='cuda'
        )
        smoke_rows.append(
            {
                'variant': variant,
                'model_uid': spec.model_uid,
                'sample_count': len(scores),
                'finite': bool(np.isfinite(scores).all()),
                'score_min': float(scores.min()),
                'score_max': float(scores.max()),
                'cuda_peak_bytes': int(torch.cuda.max_memory_allocated()),
                'device': torch.cuda.get_device_name(0),
            }
        )
        del model, scores
        gc.collect()
        torch.cuda.empty_cache()
    smoke_summary = pd.DataFrame(smoke_rows)
    display(smoke_summary)
else:
    display(Markdown('`RUN_MODEL_SMOKE=False`: checkpoint GPU smoke를 건너뜁니다.'))

## 5. 선택 단계 B — 전체 SurvFace CR-FIQA score artifact

모든 aligned face에 scalar quality를 한 번만 계산해 `sample_id, fiqa_score, fiqa_model_uid` 형태로 저장합니다. 전체 입력 bundle과 checkpoint lineage를 manifest에 고정합니다. 각 8,192장 shard를 원자적으로 기록하므로 중단 후 동일 설정으로 다시 실행하면 완료 shard부터 재개합니다. 전체 추론을 요청할 때는 쓰기 플래그도 반드시 켜야 합니다.

In [ ]:
selected_spec = CRFIQA_VARIANTS[FIQA_VARIANT]
FIQA_OUTPUT_DIR = RESULT_ROOT / 'fiqa_scores' / 'survface' / selected_spec.model_uid
fiqa_artifact = None

if RUN_FIQA_INFERENCE:
    if FIQA_OUTPUT_DIR.exists() and not OVERWRITE_OUTPUTS:
        raise FileExistsError(f'기존 FIQA artifact가 있습니다: {FIQA_OUTPUT_DIR}')
    if not WRITE_FIQA_ARTIFACT:
        raise ValueError('전체 FIQA inference에는 WRITE_FIQA_ARTIFACT=True가 필요합니다.')
    model, loaded_spec = load_cr_fiqa(
        FIQA_CHECKPOINTS[FIQA_VARIANT], variant=FIQA_VARIANT, device='cuda',
        verify_official_hash=True,
    )
    fiqa_artifact = materialize_aligned_bundle_score_artifact(
        ALIGNED_BUNDLE_DIR,
        FIQA_OUTPUT_DIR,
        model=model,
        model_uid=loaded_spec.model_uid,
        checkpoint_sha256=model.cr_fiqa_checkpoint_sha256,
        variant=loaded_spec.variant,
        batch_size=FIQA_BATCH_SIZE,
        shard_size=FIQA_SHARD_SIZE,
        device='cuda',
        overwrite=OVERWRITE_OUTPUTS,
    )
    del model
    gc.collect()
else:
    if FIQA_OUTPUT_DIR.is_dir():
        fiqa_artifact = load_fiqa_score_artifact(FIQA_OUTPUT_DIR)

if fiqa_artifact is None:
    display(Markdown(f'FIQA artifact 대기 중: `{FIQA_OUTPUT_DIR}`'))
else:
    display(pd.DataFrame([fiqa_artifact.manifest['score_summary']]))
    display(fiqa_artifact.scores.head())

## 6. 선택 단계 C — calibration PQ-ADC score만 재생

완료된 Step-4 run은 test retrieval core를 보존했지만 calibration query의 개별 compressed score는 threshold 산출 뒤 저장하지 않았습니다. 따라서 이 단계는 기존 embedding과 frozen PQ codec을 재사용해 **calibration search만** 재생합니다. 재생한 점수로 기존 다섯 target FPIR threshold가 `1e-12` 이내에서 재현되지 않으면 artifact를 만들지 않습니다.

In [ ]:
CONDITION_OUTPUT_DIR = (
    RESULT_ROOT / 'condition_scores' / str(source_manifest['run_id'])
    / f'{COMPRESSION_PROFILE}__{SEARCH_MODE}'
)
condition_tables = None

if RUN_SCORE_REPLAY:
    if CONDITION_OUTPUT_DIR.exists() and not OVERWRITE_OUTPUTS:
        raise FileExistsError(f'기존 condition score artifact가 있습니다: {CONDITION_OUTPUT_DIR}')
    if not WRITE_SCORE_ARTIFACT:
        raise ValueError('calibration score replay에는 WRITE_SCORE_ARTIFACT=True가 필요합니다.')
    in_memory_tables = replay_survface_adc_condition_scores(
        SOURCE_RUN_DIR, compression_profile=COMPRESSION_PROFILE, search_mode=SEARCH_MODE
    )
    condition_tables = write_condition_score_artifact(
        CONDITION_OUTPUT_DIR, in_memory_tables, overwrite=OVERWRITE_OUTPUTS
    )
else:
    if CONDITION_OUTPUT_DIR.is_dir():
        condition_tables = load_condition_score_artifact(CONDITION_OUTPUT_DIR)

if condition_tables is None:
    display(Markdown(f'condition score artifact 대기 중: `{CONDITION_OUTPUT_DIR}`'))
else:
    display(pd.DataFrame(condition_tables.manifest['global_threshold_reproduction']))
    display(
        pd.DataFrame(
            [{
                'condition_uid': condition_tables.condition_uid,
                'calibration_rows': len(condition_tables.calibration),
                'test_rows': len(condition_tables.test),
                'score_space': condition_tables.manifest['score_space'],
            }]
        )
    )

In [ ]:
# 7. FIQA를 calibration/test query에 완전한 one-to-one으로 결합
calibration_with_fiqa = None
test_with_fiqa = None
if fiqa_artifact is not None and condition_tables is not None:
    calibration_with_fiqa, test_with_fiqa = join_fiqa_score_artifacts(
        condition_tables, fiqa_artifact
    )
    display(
        pd.DataFrame(
            [
                {
                    'split': 'calibration', 'rows': len(calibration_with_fiqa),
                    'fiqa_min': calibration_with_fiqa['fiqa_score'].min(),
                    'fiqa_median': calibration_with_fiqa['fiqa_score'].median(),
                    'fiqa_max': calibration_with_fiqa['fiqa_score'].max(),
                },
                {
                    'split': 'test', 'rows': len(test_with_fiqa),
                    'fiqa_min': test_with_fiqa['fiqa_score'].min(),
                    'fiqa_median': test_with_fiqa['fiqa_score'].median(),
                    'fiqa_max': test_with_fiqa['fiqa_score'].max(),
                },
            ]
        )
    )
else:
    display(Markdown('FIQA와 condition score artifact가 모두 준비되면 결합합니다.'))

## 8. 선택 단계 D — Global 대 FIQA-conditioned calibration

Calibration을 `identity_id` SHA-256으로 fit/safety에 결정론적으로 분할합니다. FIQA cutpoint는 fit subset에서만 정하며, Low/High threshold는 group별 non-mated score에서 산출한 뒤 표본수 기반 shrinkage와 safety 상한을 적용합니다. Test에서는 고정된 cutpoint/threshold를 한 번만 적용합니다.

In [ ]:
CALIBRATION_OUTPUT_DIR = (
    RESULT_ROOT / 'global_vs_fiqa' / str(source_manifest['run_id'])
    / selected_spec.model_uid / f'{COMPRESSION_PROFILE}__{SEARCH_MODE}'
)
comparison = None

if RUN_THRESHOLD_CALIBRATION:
    if (
        WRITE_CALIBRATION_ARTIFACT
        and CALIBRATION_OUTPUT_DIR.exists()
        and not OVERWRITE_OUTPUTS
    ):
        raise FileExistsError(
            f'기존 calibration artifact가 있습니다: {CALIBRATION_OUTPUT_DIR}'
        )
    if calibration_with_fiqa is None or test_with_fiqa is None:
        raise RuntimeError('먼저 FIQA 및 condition score artifact를 준비해야 합니다.')
    in_memory_comparison = run_global_vs_fiqa_calibration(
        calibration_with_fiqa,
        test_with_fiqa,
        target_fpirs=TARGET_FPIRS,
        bin_count=QUALITY_BIN_COUNT,
        shrinkage_strength=SHRINKAGE_STRENGTH,
        minimum_group_non_mated=MINIMUM_GROUP_NON_MATED,
        safety_fraction=SAFETY_FRACTION,
        partition_seed=PARTITION_SEED,
        condition_manifest=condition_tables.manifest,
        fiqa_manifest=fiqa_artifact.manifest,
    )
    comparison = (
        write_calibration_comparison_artifact(
            CALIBRATION_OUTPUT_DIR, in_memory_comparison, overwrite=OVERWRITE_OUTPUTS
        )
        if WRITE_CALIBRATION_ARTIFACT
        else in_memory_comparison
    )
else:
    if CALIBRATION_OUTPUT_DIR.is_dir():
        comparison = load_calibration_comparison_artifact(CALIBRATION_OUTPUT_DIR)

if comparison is None:
    display(Markdown(f'calibration comparison 대기 중: `{CALIBRATION_OUTPUT_DIR}`'))
else:
    result_columns = [
        'target_fpir', 'method', 'realized_fpir',
        'fpir_wilson95_low', 'fpir_wilson95_high', 'tpir_at_rank_k',
        'tpir_at_rank_k_wilson95_low', 'tpir_at_rank_k_wilson95_high',
        'target_met_on_test', 'target_met_by_wilson_upper',
    ]
    display(
        comparison.method_summary[result_columns]
        .sort_values(['target_fpir', 'method']).reset_index(drop=True)
    )
    display(comparison.paired_comparisons)

## 9. Saliency 1차 목적 — 기존 압축/retrieval 진단 유지

이 셀은 새 threshold를 학습하지 않습니다. 기존 ArcFace/SurvFace 결과에서 PQ m128 ADC 조건의 공간적 saliency feature와 embedding distortion, score/rank 변화, crossing 간 연관 결과를 읽습니다. Spearman rho는 보조 진단이며 임의 threshold 가중식으로 바꾸지 않습니다.

In [ ]:
primary_saliency_views = {}
if RUN_PRIMARY_SALIENCY_SUMMARY:
    primary_saliency = load_saliency_primary_diagnostics(SOURCE_RUN_DIR)
    for name, frame in primary_saliency.items():
        mask = pd.Series(True, index=frame.index)
        if 'compression_profile' in frame:
            mask &= frame['compression_profile'].astype(str).eq(COMPRESSION_PROFILE)
        if 'search_mode' in frame:
            mask &= frame['search_mode'].astype(str).eq(SEARCH_MODE)
        selected = frame.loc[mask].copy()
        preferred = [
            'analysis_scope', 'analysis_tier', 'compression_profile', 'search_mode',
            'target_fpir', 'threshold_policy', 'is_mated', 'saliency_feature',
            'instability_predictor', 'sensitivity_metric', 'event_metric',
            'sample_count', 'paired_query_count', 'identity_count', 'event_count',
            'event_rate', 'spearman_rho', 'bootstrap_ci_low', 'bootstrap_ci_high',
            'frozen_event_count', 'frozen_event_rate', 'recalibrated_event_count',
            'recalibrated_event_rate', 'recalibrated_minus_frozen_rate',
            'resolved_event_count', 'introduced_event_count',
            'frozen_spearman_rho', 'recalibrated_spearman_rho',
            'recalibrated_minus_frozen_rho', 'paired_bootstrap_ci_low',
            'paired_bootstrap_ci_high', 'event_support_eligible',
            'association_status',
        ]
        columns = [column for column in preferred if column in selected]
        primary_saliency_views[name] = selected[columns].reset_index(drop=True)
        display(Markdown(f'**{name}** — {len(selected):,} rows'))
        if (
            selected.empty
            and SEARCH_MODE == 'pq_adc_exhaustive'
            and name in {'threshold_policy', 'threshold_policy_rho'}
        ):
            display(Markdown('PQ ADC score-space에는 frozen-origin 대 recalibrated threshold policy 비교가 적용되지 않습니다(not applicable).'))
            continue
        preview = primary_saliency_views[name]
        if 'saliency_feature' in preview and not preview.empty:
            preview = (
                preview.sort_values('saliency_feature')
                .groupby('saliency_feature', sort=True, group_keys=False)
                .head(1)
                .reset_index(drop=True)
            )
        display(preview.head(30))
else:
    display(Markdown('`RUN_PRIMARY_SALIENCY_SUMMARY=False`: 기존 saliency 진단 로드를 건너뜁니다.'))

## 10. Saliency 2차 목적 — FIQA 이후 추가정보 검증 readiness gate

사전 지정 feature는 `outside_face_attention`, `saliency_entropy` 두 개뿐입니다. High/Low saliency masking과 random masking 결과는 feature가 아니라 faithfulness/negative-control 근거로만 유지합니다. Calibration과 test 모두에서 동일 target의 saliency coverage가 95% 이상이어야 다음 노트북(`01_saliency_incremental_threshold_calibration.ipynb`, 향후 작성 대상)으로 진행할 수 있습니다. 현재 artifact는 test-only이므로 예상 결과는 `blocked`입니다.

In [ ]:
saliency_readiness = None
SALIENCY_FEATURE_PATH = (
    SOURCE_RUN_DIR / 'artifacts' / 'step2_workflow'
    / 'saliency_population' / 'saliency_features.csv'
)
if RUN_SALIENCY_READINESS_CHECK:
    if condition_tables is None:
        raise RuntimeError('readiness 계산에는 condition score artifact가 먼저 필요합니다.')
    saliency_features = pd.read_csv(
        SALIENCY_FEATURE_PATH,
        usecols=[
            'sample_id', 'saliency_target_name', 'heatmap_available',
            'gradcam_valid_heatmap', 'outside_face_attention', 'saliency_entropy',
        ],
        low_memory=False,
    )
    saliency_readiness = assess_saliency_incremental_readiness(
        condition_tables.calibration,
        condition_tables.test,
        saliency_features,
        requested_features=('outside_face_attention', 'saliency_entropy'),
        minimum_coverage=0.95,
    )
    display(pd.DataFrame([saliency_readiness.as_dict()]))
    if not saliency_readiness.secondary_calibration_supported:
        display(Markdown('**차단됨:** calibration saliency 없이 FIQA+Saliency threshold를 fit하면 test leakage가 되므로 실행하지 않습니다.'))
else:
    display(Markdown('`RUN_SALIENCY_READINESS_CHECK=False`: 대용량 saliency feature 파일을 읽지 않습니다. 현재 알려진 calibration coverage는 0이므로 2차 calibration은 차단 상태입니다.'))

In [ ]:
# 11. 현재 상태 요약 — 계산 완료와 미실행을 명확히 구분
status_rows = [
    {'stage': 'checkpoint_preflight', 'status': 'validated', 'artifact': str(FIQA_CHECKPOINTS[FIQA_VARIANT])},
    {'stage': 'fiqa_scores', 'status': 'available' if fiqa_artifact is not None else 'not_run', 'artifact': str(FIQA_OUTPUT_DIR)},
    {'stage': 'calibration_score_replay', 'status': 'available' if condition_tables is not None else 'not_run', 'artifact': str(CONDITION_OUTPUT_DIR)},
    {'stage': 'global_vs_fiqa', 'status': 'available' if comparison is not None else 'not_run', 'artifact': str(CALIBRATION_OUTPUT_DIR)},
    {'stage': 'saliency_primary_analysis', 'status': 'available' if primary_saliency_views else 'not_loaded', 'artifact': str(SOURCE_RUN_DIR / 'artifacts' / 'step2_workflow')},
    {
        'stage': 'fiqa_plus_saliency',
        'status': saliency_readiness.status if saliency_readiness is not None else 'blocked_pending_calibration_saliency',
        'artifact': 'not_created',
    },
]
status_table = pd.DataFrame(status_rows)
display(status_table)

## 12. 해석 규칙

- 결론은 `realized_fpir`, Wilson 95% CI, `tpir_at_rank_k`(현재 Rank-20), `target_met_on_test`, paired bootstrap 차이를 함께 봅니다.
- `global_safe`와 `shrunk_safe`는 held-out 경험적 보수화이며 formal FPIR guarantee가 아닙니다.
- FIQA가 아주 조금 좋아졌다는 이유만으로 채택하지 않습니다. 사전 지정한 여러 target FPIR에서 방향이 재현되고, paired CI와 TPIR 손실까지 검토해야 합니다.
- FIQA가 Global보다 낫지 않아도 saliency의 1차 연구 질문은 독립적으로 유지됩니다.
- FIQA+Saliency는 calibration saliency를 별도로 생성한 뒤에만 시험하며, test 기반 feature 선택이나 threshold 재조정은 금지합니다.
- 이 노트북에서 생성한 compact artifact만 이후 `00_cross_dataset_results.ipynb`의 입력 후보가 됩니다. 공통 보고 노트북 연결은 결과가 실제로 생성·검증된 뒤 별도 변경으로 수행합니다.